# 宠物慢领养风险预测 — 数据分析与建模

> **原始问题**: 根据宠物的信息，预测它多久会被领养。
>
> **最终问题**: 预测一只宠物是否有慢领养风险（二分类）。


---

## 第 1 步：读取和理解数据

先别急着建模，先知道自己手里有什么数据。


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, accuracy_score)
from sklearn.inspection import permutation_importance

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

print("库导入完成 ✓")


In [ ]:
# 加载数据
train = pd.read_csv("../data/raw/train.csv")

# 加载标签映射表
breed_labels = pd.read_csv("../data/raw/BreedLabels.csv")
color_labels = pd.read_csv("../data/raw/ColorLabels.csv")
state_labels = pd.read_csv("../data/raw/state_labels.csv")

print("数据加载完成 ✓")
print(f"\n数据维度: {train.shape}")
print(f"行数: {train.shape[0]}, 列数: {train.shape[1]}")
train.head()


In [ ]:
# 查看数据结构
print("=== 列信息 ===")
train.info()

print("\n=== 缺失值统计 ===")
missing = train.isnull().sum()
missing_pct = (train.isnull().sum() / len(train) * 100).round(2)
missing_df = pd.DataFrame({
    '缺失数量': missing,
    '缺失比例(%)': missing_pct
})
missing_df = missing_df[missing_df['缺失数量'] > 0].sort_values('缺失数量', ascending=False)
print(missing_df if len(missing_df) > 0 else "仅 Name 和 Description 有缺失，其他字段完整")


**发现**:
- 总共 **14,993** 条记录
- 原始数据有 **24 列**
- 大部分结构化字段没有缺失
- **Name** 和 **Description** 有缺失值
- 目标变量是 **AdoptionSpeed**（0–4）


---

## 第 2 步：理解目标变量 AdoptionSpeed

- 0 = 当天被领养
- 1 = 1–7 天内被领养
- 2 = 8–30 天内被领养
- 3 = 31–90 天内被领养
- 4 = 100 天后仍未被领养


In [ ]:
# 目标变量分布
print("AdoptionSpeed 各类别数量:")
print(train["AdoptionSpeed"].value_counts().sort_index())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 柱状图
speed_counts = train["AdoptionSpeed"].value_counts().sort_index()
labels = ['0: 当天', '1: 1-7天', '2: 8-30天', '3: 31-90天', '4: 未领养']
axes[0].bar(labels, speed_counts.values, color='steelblue', edgecolor='white')
axes[0].set_title("AdoptionSpeed 各类别数量", fontsize=14)
axes[0].set_ylabel("数量")
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(speed_counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontsize=11)

# 饼图
axes[1].pie(speed_counts.values, labels=labels, autopct='%1.1f%%',
            colors=sns.color_palette("Blues", 5), startangle=90,
            textprops={'fontsize': 9})
axes[1].set_title("AdoptionSpeed 各类别占比", fontsize=14)

plt.tight_layout()
plt.show()


**关键发现**:
- **AdoptionSpeed = 0**（当天被领养）的样本很少
- **AdoptionSpeed = 2** 和 **4** 的样本比较多
- 目标变量**不平衡**，AdoptionSpeed = 0 太少，模型很难学

> 如果后面模型预测不好，可能不是代码错，而是这个任务本身有难度。


---

## 第 3 步：数据清洗

不做大规模删除，做比较稳妥的清洗。


### 3.1 处理缺失值


In [ ]:
# 复制一份用于清洗，保留原始数据
train_clean = train.copy()

# Description: 用空字符串填充，并创建描述长度特征
train_clean["Description"] = train_clean["Description"].fillna("")
train_clean["DescriptionLength"] = train_clean["Description"].str.len()

print(f"Description 缺失值已填充")
print(f"DescriptionLength 范围: {train_clean['DescriptionLength'].min()} – {train_clean['DescriptionLength'].max()}")
print(f"DescriptionLength 中位数: {train_clean['DescriptionLength'].median():.0f}")

# 思路: 不直接让模型理解整段文字，先用"描述长度"作为一个简单特征
print("\n✓ 创建特征: DescriptionLength")


### 3.2 识别"隐性缺失"：Name 字段


In [ ]:
# 检查 Name 字段的特殊值
# "No Name Yet" 不是 NaN，但现实含义是"没有名字"
no_name_count = (train_clean["Name"] == "No Name Yet").sum()
nan_name_count = train_clean["Name"].isna().sum()

print(f"'No Name Yet' 的数量: {no_name_count}")
print(f"Name 为 NaN 的数量: {nan_name_count}")

# 创建 HasName 特征
train_clean["HasName"] = train_clean["Name"].apply(
    lambda x: 0 if pd.isna(x) or x == "No Name Yet" else 1
)

print(f"\nHasName 分布:")
print(f"  有名字: {train_clean['HasName'].sum()} ({train_clean['HasName'].mean():.1%})")
print(f"  无名字: {(train_clean['HasName'] == 0).sum()} ({(1 - train_clean['HasName'].mean()):.1%})")

print("\n✓ 创建特征: HasName")
print("思路: 缺失值不一定只表现为 NaN，有些是'隐性缺失'")


### 3.3 检查重复值


In [ ]:
# 完全重复检查
n_dup = train_clean.duplicated().sum()
print(f"完全重复的行数: {n_dup}")
print("结论: 没有完全重复的数据")

# 近似重复检查（忽略 Name、PetID、PhotoAmt）
approx_dup_cols = [c for c in train_clean.columns if c not in ["Name", "PetID", "PhotoAmt"]]
n_approx_dup = train_clean.duplicated(subset=approx_dup_cols).sum()
print(f"\n忽略 Name/PetID/PhotoAmt 后的近似重复: {n_approx_dup} 条")

if n_approx_dup > 0:
    print("\n注意: 这些可能是同一个救助者发布的一窝相似宠物")
    print("不一定是真正的重复数据，所以**不自动删除**")

print("\n思路: 代码只能帮我们找嫌疑，不能直接替我们判案")


### 3.4 年龄异常值分析


In [ ]:
# 年龄基础统计
print("=== 年龄 (Age) 统计 ===")
print(f"最小值: {train_clean['Age'].min()} 个月")
print(f"最大值: {train_clean['Age'].max()} 个月")
print(f"平均值: {train_clean['Age'].mean():.1f} 个月")
print(f"中位数: {train_clean['Age'].median():.1f} 个月")
print(f"标准差: {train_clean['Age'].std():.1f} 个月")

# 箱线图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 全量箱线图
axes[0].boxplot(train_clean["Age"], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[0].set_title("年龄分布 — 箱线图（全量）", fontsize=14)
axes[0].set_ylabel("年龄（月）")
axes[0].set_xticklabels(["Age"])

# 按 AdoptionSpeed 的箱线图
age_by_speed = [train_clean[train_clean["AdoptionSpeed"] == i]["Age"].values for i in range(5)]
bp = axes[1].boxplot(age_by_speed, labels=labels, patch_artist=True,
                      boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title("按领养速度的年龄分布", fontsize=14)
axes[1].set_ylabel("年龄（月）")
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print("\n发现: Age 最大值 255 个月看起来离谱，但不一定全部是错")
print("思路: 对异常值不要一刀切，先判断它对分析有没有影响")


---

## 第 4 步：EDA 探索性数据分析

目标：看哪些因素可能和领养速度有关。围绕具体问题展开，不是随便画图。


### 4.1 猫狗类型 Type — 不同物种的领养速度是否不同？


In [ ]:
# Type 分布
type_map = {1: '狗 (Dog)', 2: '猫 (Cat)'}
train_clean['TypeName'] = train_clean['Type'].map(type_map)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 数量图
for pet_type, name, color in [(1, '狗', '#FF8C00'), (2, '猫', '#9370DB')]:
    subset = train_clean[train_clean['Type'] == pet_type]
    speed_dist = subset['AdoptionSpeed'].value_counts().sort_index()
    axes[0].bar([f"{name}\n{l}" for l in labels], speed_dist.values,
                color=color, alpha=0.7, edgecolor='white', label=name)

axes[0].set_title("不同宠物类型的领养速度数量分布", fontsize=14)
axes[0].set_ylabel("数量")
axes[0].legend()
axes[0].tick_params(axis='x', rotation=30)

# 比例图
for pet_type, name, color in [(1, '狗', '#FF8C00'), (2, '猫', '#9370DB')]:
    subset = train_clean[train_clean['Type'] == pet_type]
    speed_pct = subset['AdoptionSpeed'].value_counts(normalize=True).sort_index() * 100
    axes[1].bar([f"{name}\n{l}" for l in labels], speed_pct.values,
                color=color, alpha=0.7, edgecolor='white', label=name)

axes[1].set_title("不同宠物类型的领养速度比例分布", fontsize=14)
axes[1].set_ylabel("比例 (%)")
axes[1].legend()
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

# 具体数字
print("\n狗: AdoptionSpeed=4 占比:",
      (train_clean[train_clean['Type'] == 1]['AdoptionSpeed'] == 4).mean():.2%")
print("猫: AdoptionSpeed=4 占比:",
      (train_clean[train_clean['Type'] == 2]['AdoptionSpeed'] == 4).mean():.2%")

print("\n发现: 狗在较慢类别中比例更高，猫在较快类别中比例相对更高")
print("但不能只看 Type，品种、年龄、照片数量也会影响结果")
print("Type 可能有关系，但不是唯一原因")


### 4.2 年龄 Age — 年龄会不会影响领养速度？


In [ ]:
# 按领养速度看年龄统计
age_stats = train_clean.groupby("AdoptionSpeed")["Age"].agg(["mean", "median", "count"])
age_stats.columns = ["平均年龄", "中位年龄", "样本数"]
age_stats["平均年龄"] = age_stats["平均年龄"].round(1)
age_stats["中位年龄"] = age_stats["中位年龄"].round(1)
print("按 AdoptionSpeed 的年龄统计:")
print(age_stats)

# 观察: mean 和 median 差很多
print(f"\n总体 mean/median 差异: {train_clean['Age'].mean():.1f} vs {train_clean['Age'].median():.1f}")
print("说明年龄分布很偏，有高龄离群点在拉高均值")

# 按年龄组分析
bins = [0, 3, 12, 24, 48, 120, 300]
age_labels = ['0-3个月', '4-12个月', '1-2岁', '2-4岁', '4-10岁', '10岁以上']
train_clean["AgeGroup"] = pd.cut(train_clean["Age"], bins=bins, labels=age_labels)

# 每个年龄组的平均 AdoptionSpeed 和慢领养比例
age_analysis = train_clean.groupby("AgeGroup", observed=False).agg(
    平均AdoptionSpeed=("AdoptionSpeed", "mean"),
    慢领养比例=("AdoptionSpeed", lambda x: (x >= 3).mean()),
    样本数=("AdoptionSpeed", "count")
).round(3)
print("\n按年龄组的分析:")
print(age_analysis)

# 可视化
fig, ax = plt.subplots(figsize=(10, 5))
age_analysis["慢领养比例"].plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title("各年龄组的慢领养比例", fontsize=14)
ax.set_ylabel("慢领养比例")
ax.set_xlabel("年龄组")
for i, (v, c) in enumerate(zip(age_analysis['慢领养比例'], age_analysis['样本数'])):
    ax.text(i, v + 0.01, f'{v:.2%}\n(n={c})', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print("\n发现:")
print("- 0-3 个月幼宠明显更不容易慢领养")
print("- 4 个月以后差异没有稳定递增")
print("结论: 年龄有影响，但不是线性关系")


### 4.3 照片数量 PhotoAmt — 照片多是否更容易被领养？


In [ ]:
# 照片数量分析
print("=== PhotoAmt 分析 ===")

# 照片分组
train_clean["PhotoGroup"] = pd.cut(
    train_clean["PhotoAmt"],
    bins=[-1, 0, 1, 3, 6, 10, 100],
    labels=["0张", "1张", "2-3张", "4-6张", "7-10张", "10+张"]
)

photo_analysis = train_clean.groupby("PhotoGroup", observed=False).agg(
    平均AdoptionSpeed=("AdoptionSpeed", "mean"),
    慢领养比例=("AdoptionSpeed", lambda x: (x >= 3).mean()),
    样本数=("AdoptionSpeed", "count")
).round(3)
print(photo_analysis)

# 可视化
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(photo_analysis)), photo_analysis["慢领养比例"].values,
       color='steelblue', edgecolor='white')
ax.set_xticks(range(len(photo_analysis)))
ax.set_xticklabels(photo_analysis.index)
ax.set_title("照片数量与慢领养比例", fontsize=14)
ax.set_ylabel("慢领养比例")
for i, (v, c) in enumerate(zip(photo_analysis['慢领养比例'], photo_analysis['样本数'])):
    ax.text(i, v + 0.01, f'{v:.2%}\n(n={c})', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print("\n发现:")
print("- 0 张照片的宠物平均领养更慢")
print("- 1 张以上明显好很多")
print("- 4-6 张左右效果最好")
print("- 7+ 张没有继续明显改善")
print("\n结论: 有照片很重要，但照片数量达到一定程度后边际收益变小")
print("不要为了迎合直觉强行写线性结论，要看数据到底支持什么")


### 4.4 Fee 领养费用 — 收费会不会影响领养速度？


In [ ]:
# Fee 分析
print("=== Fee 分析 ===")

fee_stats = train_clean.groupby("AdoptionSpeed")["Fee"].agg(["mean", "median", "std", "count"]).round(1)
print(fee_stats)

# Free vs Paid
train_clean["IsFree"] = (train_clean["Fee"] == 0).astype(int)
free_stats = train_clean.groupby("IsFree")["AdoptionSpeed"].agg(["mean", "count"]).round(3)
free_stats.columns = ["平均AdoptionSpeed", "数量"]
free_stats.index = ["付费", "免费"]
print(f"\n{free_stats}")

print("\n发现:")
print("- 所有类别的 Fee 中位数都是 0")
print("- 平均值差别也很小")
print("- Free vs Paid 的差别不明显")
print("结论: Fee 在这个数据集中**不是强影响因素**")
print("思路: 有些变量看起来可能有用，但数据不支持，就要承认它不明显")


### 4.5 Vaccinated / Dewormed / Sterilized — 健康相关变量


In [ ]:
# 健康状态与领养速度
health_cols = ['Vaccinated', 'Dewormed', 'Sterilized']
status_map = {1: 'Yes', 2: 'No', 3: 'Not Sure'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, col in enumerate(health_cols):
    stats = train_clean.groupby(col)['AdoptionSpeed'].mean()
    axes[i].bar([f'{status_map.get(k, k)}' for k in stats.index],
                stats.values, color='steelblue', edgecolor='white')
    axes[i].set_title(f'{col} — 平均 AdoptionSpeed', fontsize=13)
    axes[i].set_ylabel('平均 AdoptionSpeed')
    for j, v in enumerate(stats.values):
        axes[i].text(j, v + 0.02, f'{v:.2f}', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

# 乍看反直觉：未接种疫苗的宠物看起来更快被领养？
print("⚠ 初步观察: 未接种疫苗/未驱虫/未绝育的宠物似乎平均领养更快")
print("这是否反直觉？需要检查混杂因素")


### 4.5 续：检查年龄混杂因素


In [ ]:
# 按接种/驱虫/绝育状态看年龄分布
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, col in enumerate(health_cols):
    age_by_status = []
    labels_list = []
    for status in sorted(train_clean[col].unique()):
        age_by_status.append(train_clean[train_clean[col] == status]['Age'].values)
        labels_list.append(status_map.get(status, status))

    bp = axes[i].boxplot(age_by_status, labels=labels_list, patch_artist=True,
                          boxprops=dict(facecolor='steelblue', alpha=0.6))
    axes[i].set_title(f'{col} 各状态下的年龄分布', fontsize=13)
    axes[i].set_ylabel('年龄（月）')

plt.tight_layout()
plt.show()

# 数值验证
print("=== 各健康状态下的平均年龄 ===")
for col in health_cols:
    print(f"\n--- {col} ---")
    for status in sorted(train_clean[col].unique()):
        avg_age = train_clean[train_clean[col] == status]['Age'].mean()
        count = (train_clean[col] == status).sum()
        print(f"  {status_map.get(status, status)}: 平均年龄 {avg_age:.1f} 月 (n={count})")

print("\n发现:")
print("- Vaccinated = No 的平均年龄明显更小（幼宠）")
print("- Dewormed = No 的平均年龄也明显更小")
print("- Sterilized = No 也和低龄强相关")
print("\n结论: 这些变量都存在**年龄混杂因素**")
print("思路: 看到反直觉结果，不是马上说数据错，而是检查是否有混杂因素")


### 4.5 续：在同一个年龄组里比较


In [ ]:
# 在同一个 AgeGroup 里比较 Sterilized 的影响
print("=== 按年龄组分层比较绝育状态 ===")
for ag in ['0-3个月', '4-12个月', '2-4岁', '4-10岁']:
    subset = train_clean[train_clean['AgeGroup'] == ag]
    sterilized_stats = subset.groupby('Sterilized').agg(
        数量=('AdoptionSpeed', 'count'),
        平均AdoptionSpeed=('AdoptionSpeed', 'mean'),
        慢领养比例=('AdoptionSpeed', lambda x: (x >= 3).mean())
    ).round(3)
    if len(sterilized_stats) >= 2:
        print(f"\n年龄组: {ag} (n={len(subset)})")
        print(sterilized_stats)

print("\n结论:")
print("- Vaccinated 和 Dewormed 的独立影响不稳定")
print("- Sterilized 在同年龄组里有一定稳定差异")
print("- 但仍然不能解释为因果")


### 4.6 Quantity — 多只宠物会不会更难领养？


In [ ]:
# Quantity 分布
print("=== Quantity 分布 ===")
print(train_clean['Quantity'].value_counts().sort_index().head(10))

# Single vs Multiple
train_clean["SinglePet"] = (train_clean["Quantity"] == 1).astype(int)

qty_stats = train_clean.groupby("SinglePet")["AdoptionSpeed"].agg(
    ["mean", "count", lambda x: (x >= 3).mean()]
).round(3)
qty_stats.columns = ["平均AdoptionSpeed", "数量", "慢领养比例"]
qty_stats.index = ["多只发布", "单只发布"]
print(f"\n{qty_stats}")

# 可视化
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(qty_stats.index, qty_stats["慢领养比例"].values,
       color=['tomato', 'steelblue'], edgecolor='white')
ax.set_title("单只 vs 多只发布的慢领养比例", fontsize=14)
for i, v in enumerate(qty_stats["慢领养比例"].values):
    ax.text(i, v + 0.01, f'{v:.2%}', ha='center', fontsize=12)
plt.show()

print("\n发现: 多只发布可能有更高慢领养风险，但影响不算特别强")
print("思路: 不一定每个变量都要深挖，有些变量做简单验证就够了")


---

## 第 5 步：建立五分类 Baseline 模型

EDA 完成后，开始建模。先做一个 baseline，看原始任务到底难不难。


In [ ]:
# 准备建模特征
# 选择结构化的数值和类别特征
feature_cols = [
    'Type', 'Age', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2', 'Color3',
    'MaturitySize', 'FurLength', 'Vaccinated', 'Dewormed', 'Sterilized',
    'Health', 'Quantity', 'Fee', 'VideoAmt', 'PhotoAmt',
    'DescriptionLength', 'HasName'
]

model_data = train_clean[feature_cols + ['AdoptionSpeed', 'State']].dropna()

X = model_data[feature_cols]
y = model_data['AdoptionSpeed']

# state 编码（简单标签编码）
from sklearn.preprocessing import LabelEncoder
if 'State' in model_data.columns:
    X = X.copy()
    # State 已经在 feature_cols 外面，先单独处理
    pass

print(f"建模数据: {X.shape}")
print(f"y 分布:\n{y.value_counts().sort_index()}")


In [ ]:
# 划分训练/测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"训练集: {X_train.shape}")
print(f"测试集: {X_test.shape}")
print(f"\n训练集类别分布:\n{y_train.value_counts(normalize=True).sort_index().round(3)}")
print(f"\n测试集类别分布:\n{y_test.value_counts(normalize=True).sort_index().round(3)}")

# stratify=y 保持了类别比例一致 ✓


In [ ]:
# 五分类 Baseline: RandomForest
rf5 = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)
rf5.fit(X_train, y_train)

y_pred5 = rf5.predict(X_test)
y_proba5 = rf5.predict_proba(X_test)

print("=== 五分类 Baseline 评估 ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred5):.4f}")
print(f"\n分类报告:")
print(classification_report(y_test, y_pred5,
      target_names=['0:当天', '1:1-7天', '2:8-30天', '3:31-90天', '4:未领养']))


In [ ]:
# 归一化混淆矩阵 — 把每一行加起来 = 100%
cm5 = confusion_matrix(y_test, y_pred5)
cm5_normalized = cm5.astype('float') / cm5.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(9, 7))
sns.heatmap(cm5_normalized, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=['0', '1', '2', '3', '4'],
            yticklabels=['0:当天', '1:1-7天', '2:8-30天', '3:31-90天', '4:未领养'])
plt.title("五分类归一化混淆矩阵 (行=真实, 列=预测)", fontsize=14)
plt.xlabel("预测类别")
plt.ylabel("真实类别")
plt.show()

print("\n发现:")
print("- AdoptionSpeed = 4 预测最好")
print("- AdoptionSpeed = 0 很难预测，常被错分成 1")
print("- 1、2、3 中间类别经常混淆")
print("\n结论: 五分类任务太细，类别边界模糊，模型表现有限 (accuracy ≈ 0.40)")


---

## 第 6 步：尝试 One-Hot Encoding 优化五分类

很多列虽然是数字，但其实是类别（Breed1、Gender 等），不应该被当成数值。


In [ ]:
# 尝试 One-Hot Encoding
# 需要 One-Hot 的列
categorical_cols = [
    'Type', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2', 'Color3',
    'MaturitySize', 'FurLength', 'Vaccinated', 'Dewormed', 'Sterilized', 'Health'
]

# 这些列直接保留数值
numeric_cols = ['Age', 'Quantity', 'Fee', 'VideoAmt', 'PhotoAmt', 'DescriptionLength', 'HasName']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)

pipeline5 = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42,
                                           class_weight='balanced', n_jobs=-1))
])

pipeline5.fit(X_train, y_train)
y_pred5_oh = pipeline5.predict(X_test)

print("=== One-Hot Encoding 五分类评估 ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred5_oh):.4f}")
print(f"\n分类报告:")
print(classification_report(y_test, y_pred5_oh,
      target_names=['0:当天', '1:1-7天', '2:8-30天', '3:31-90天', '4:未领养']))

print("\n结论: One-Hot + RandomForest 在当前设置下没有提升")
print("accuracy 反而变成约 39%")
print("\n思路: 优化不是一定成功，机器学习就是尝试、比较、保留更合理的方向")


---

## 第 7 步：重新定义问题为二分类

> 这是项目里**最关键的一次转向**。

五分类的 1、2、3 很容易混淆，改成二分类更现实、更稳定、也更容易解释。


In [ ]:
# 定义慢领养
# Not Slow: AdoptionSpeed = 0, 1, 2
# Slow:     AdoptionSpeed = 3, 4
train_clean["SlowAdoption"] = train_clean["AdoptionSpeed"].apply(
    lambda x: 1 if x >= 3 else 0
)

slow_counts = train_clean["SlowAdoption"].value_counts()
print("=== 二分类目标分布 ===")
print(f"Not Slow (0): {slow_counts.get(0)} ({slow_counts.get(0)/len(train_clean):.1%})")
print(f"Slow (1):     {slow_counts.get(1)} ({slow_counts.get(1)/len(train_clean):.1%})")

# 可视化
fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(['Not Slow\n(0-2: ≤30天)', 'Slow\n(3-4: >30天)'],
       [slow_counts.get(0), slow_counts.get(1)],
       color=['steelblue', 'tomato'], edgecolor='white')
ax.set_title("二分类目标分布", fontsize=14)
ax.set_ylabel("数量")
for i, v in enumerate([slow_counts.get(0), slow_counts.get(1)]):
    ax.text(i, v + 100, f'{v}\n({v/len(train_clean):.1%})', ha='center', fontsize=12)
plt.tight_layout()
plt.show()

print("\n问题转变为: 预测宠物是否有慢领养风险（>30天）")
print("这比精确预测几天更现实、更稳定、也更容易解释")


---

## 第 8 步：训练二分类模型


In [ ]:
# 准备二分类数据
y_binary = train_clean.loc[X.index, "SlowAdoption"]

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

# 训练二分类 RandomForest
rf_binary = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)
rf_binary.fit(X_train_b, y_train_b)

y_pred_b = rf_binary.predict(X_test_b)
y_proba_b = rf_binary.predict_proba(X_test_b)[:, 1]

print("=== 二分类模型评估 ===")
print(f"Accuracy:  {accuracy_score(y_test_b, y_pred_b):.4f}")
print(f"\n分类报告:")
print(classification_report(y_test_b, y_pred_b,
      target_names=['Not Slow', 'Slow']))

# 数值验证
from sklearn.metrics import precision_score, recall_score, f1_score
print(f"Precision: {precision_score(y_test_b, y_pred_b):.4f}")
print(f"Recall:    {recall_score(y_test_b, y_pred_b):.4f}")
print(f"F1-score:  {f1_score(y_test_b, y_pred_b):.4f}")
print("\n比五分类更稳定，accuracy ≈ 0.67")


In [ ]:
# 二分类混淆矩阵
cm_b = confusion_matrix(y_test_b, y_pred_b)
cm_b_pct = cm_b.astype('float') / cm_b.sum(axis=1)[:, np.newaxis] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 数量
sns.heatmap(cm_b, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Not Slow', 'Slow'], yticklabels=['Not Slow', 'Slow'])
axes[0].set_title("混淆矩阵 (数量)", fontsize=14)
axes[0].set_xlabel("预测")
axes[0].set_ylabel("真实")

# 百分比
sns.heatmap(cm_b_pct, annot=True, fmt='.1f', cmap='Blues', ax=axes[1],
            xticklabels=['Not Slow', 'Slow'], yticklabels=['Not Slow', 'Slow'])
axes[1].set_title("混淆矩阵 (行百分比)", fontsize=14)
axes[1].set_xlabel("预测")
axes[1].set_ylabel("真实")

plt.tight_layout()
plt.show()

tp = cm_b[1, 1]
tn = cm_b[0, 0]
fp = cm_b[0, 1]
fn = cm_b[1, 0]
print(f"True Positive (正确预测慢领养): {tp}")
print(f"True Negative (正确预测非慢领养): {tn}")
print(f"False Positive (误报慢领养): {fp}")
print(f"False Negative (漏报慢领养): {fn}")
print(f"\n真实 Not Slow 中，{cm_b_pct[0,0]:.1f}% 被预测对")
print(f"真实 Slow 中，{cm_b_pct[1,1]:.1f}% 被预测对")


---

## 第 9 步：ROC-AUC 评估

不只看 accuracy，还要看模型的风险排序能力。


In [ ]:
# ROC 曲线
fpr, tpr, thresholds = roc_curve(y_test_b, y_proba_b)
auc_score = roc_auc_score(y_test_b, y_proba_b)

plt.figure(figsize=(8, 7))
plt.plot(fpr, tpr, color='darkorange', lw=2.5,
         label=f'RandomForest (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--',
         label=f'随机猜测 (AUC = 0.500)')
plt.fill_between(fpr, tpr, alpha=0.15, color='darkorange')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("假阳性率 (False Positive Rate)", fontsize=13)
plt.ylabel("真阳性率 (True Positive Rate)", fontsize=13)
plt.title("ROC 曲线 — 慢领养风险预测", fontsize=15)
plt.legend(loc='lower right', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"ROC-AUC: {auc_score:.4f}")
print("\n解读:")
print("- 随机抽一只慢领养宠物和一只非慢领养宠物")
print(f"- 模型有 {auc_score*100:.1f}% 的概率给慢领养宠物更高风险分数")
print("- ROC 曲线明显高于随机猜测线")
print("\n结论: 二分类模型不仅要看预测对错，还要看能不能把高风险样本排在前面")
print("当前模型有区分能力 ✓")


---

## 第 10 步：特征重要性分析

建模后不能只看分数，还要解释模型主要依赖什么。


In [ ]:
# 随机森林自带的特征重要性
default_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_binary.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
top15 = default_importance.head(15)
colors = plt.cm.Blues(np.linspace(0.35, 0.85, 15))
ax.barh(range(15), top15['importance'].values, color=colors, edgecolor='white')
ax.set_yticks(range(15))
ax.set_yticklabels(top15['feature'].values)
ax.invert_yaxis()
ax.set_xlabel("Feature Importance", fontsize=13)
ax.set_title("特征重要性 (RandomForest 自带)", fontsize=14)
for i, v in enumerate(top15['importance'].values):
    ax.text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print("注意: DescriptionLength 很高，但随机森林自带重要性可能偏向连续变量")
print("需要进一步验证 →")


### 10.2 Permutation Importance — 更稳健的评估

逻辑：把某一列打乱，如果模型表现明显下降，说明这个特征重要。


In [ ]:
# Permutation Importance
perm_result = permutation_importance(
    rf_binary, X_test_b, y_test_b,
    n_repeats=10, random_state=42, n_jobs=-1, scoring='accuracy'
)

perm_importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance_mean': perm_result.importances_mean,
    'importance_std': perm_result.importances_std
}).sort_values('importance_mean', ascending=False)

# 可视化
fig, ax = plt.subplots(figsize=(10, 8))
top_perm = perm_importance_df.head(15)

ax.barh(range(15), top_perm['importance_mean'].values[::-1],
        color='steelblue', edgecolor='white', xerr=top_perm['importance_std'].values[::-1],
        capsize=3)
ax.set_yticks(range(15))
ax.set_yticklabels(top_perm['feature'].values[::-1])
ax.invert_yaxis()
ax.set_xlabel("重要性下降 (Accuracy)", fontsize=13)
ax.set_title("Permutation Importance (打乱某列后 accuracy 下降量)", fontsize=14)

for i, v in enumerate(top_perm['importance_mean'].values[::-1]):
    ax.text(v + 0.0005, i, f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("=== Permutation Importance 排名 ===")
print(perm_importance_df[['feature', 'importance_mean']].head(10).to_string(index=False))

print("\n结论:")
print("- Age 最重要")
print("- Breed1 第二重要")
print("- PhotoAmt 第三")
print("- DescriptionLength 第四（但比自带重要性合理）")
print("- Sterilized / FurLength / Breed2 / MaturitySize 也有一定作用")


---

## 第 11 步：深入理解 Breed1 — 为什么品种很重要？


In [ ]:
# 计算每个品种的慢领养比例
breed_analysis = train_clean.groupby("Breed1").agg(
    Count=("SlowAdoption", "size"),
    SlowRate=("SlowAdoption", "mean")
).reset_index()
breed_analysis.columns = ["BreedID", "样本数", "慢领养比例"]

# 合并品种名
breed_analysis = breed_analysis.merge(
    breed_labels, left_on="BreedID", right_on="BreedID", how="left"
)

# 只看样本数 >= 50 的品种，结果更可靠
breed_analysis_filtered = breed_analysis[breed_analysis["样本数"] >= 50].copy()
breed_analysis_filtered = breed_analysis_filtered.sort_values("慢领养比例", ascending=False)

# 慢领养比例最高的品种
print("=== 慢领养比例最高的品种 (n≥50) ===")
print(breed_analysis_filtered[["BreedName", "Type", "样本数", "慢领养比例"]].head(10).to_string(index=False))

print("\n=== 慢领养比例最低的品种 (n≥50) ===")
print(breed_analysis_filtered[["BreedName", "Type", "样本数", "慢领养比例"]].tail(10).sort_values("慢领养比例").to_string(index=False))

# 关键品种
key_breeds = ["Mixed Breed", "Poodle", "Persian", "Shih Tzu", "Golden Retriever"]
print("\n=== 关键品种的慢领养比例 ===")
for b in key_breeds:
    row = breed_analysis[breed_analysis["BreedName"] == b]
    if len(row) > 0:
        r = row.iloc[0]
        pet_type = "狗" if r["Type"] == 1 else "猫"
        print(f"  {b} ({pet_type}): {r['慢领养比例']:.2%} (n={int(r['样本数'])})")


In [ ]:
# 可视化: 样本数 >= 50 的品种慢领养比例
plot_data = breed_analysis_filtered.sort_values("慢领养比例")

fig, ax = plt.subplots(figsize=(12, 20))
colors = ['tomato' if v > breed_analysis_filtered["慢领养比例"].median() else 'steelblue'
          for v in plot_data["慢领养比例"]]

ax.barh(range(len(plot_data)), plot_data["慢领养比例"].values, color=colors, edgecolor='white')
ax.set_yticks(range(len(plot_data)))
ax.set_yticklabels([f"{n} ({t})" for n, t in zip(plot_data["BreedName"], plot_data["样本数"])], fontsize=9)
ax.axvline(plot_data["慢领养比例"].median(), color='red', linestyle='--',
           label=f'中位数: {plot_data["慢领养比例"].median():.1%}')
ax.set_xlabel("慢领养比例", fontsize=13)
ax.set_title("各品种慢领养比例 (n≥50)", fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()


**Breed1 分析结论:**

不同品种的慢领养比例确实存在差异:
- **Mixed Breed** 慢领养比例较高，且样本量很大
- **Poodle / Persian / Shih Tzu / Golden Retriever** 等慢领养比例较低

这解释了为什么模型认为 **Breed1 很重要**。

> ⚠ 强调: 这只是**相关性**，不是因果。不能推导出"品种决定领养速度"。


---

## 总结

### 项目全流程回顾

| 步骤 | 做什么 | 核心收获 |
|------|-------|---------|
| 1. 读数据 | train.head(), .info() | 知道自己手里有什么 |
| 2. 看目标 | value_counts() + 图 | 数据不平衡，任务有难度 |
| 3. 数据清洗 | 缺失/重复/异常值 | 不盲目删，不自动判案 |
| 4. EDA | 6 个维度 + 混杂分析 | 关联不是因果，混淆要验证 |
| 5. 五分类 baseline | RandomForest | accuracy ≈ 0.40，任务很难 |
| 6. 尝试优化 | One-Hot + Pipeline | 没提升 (accuracy ≈ 0.39)，放弃 |
| 7. 转换问题 | 二分类: Slow = 3,4 | 最关键转向 |
| 8. 二分类模型 | RandomForest n=200 | accuracy ≈ 0.67，更稳定 |
| 9. ROC-AUC | 曲线 + AUC | AUC ≈ 0.727，有区分能力 |
| 10. 特征重要性 | Permutation | Age > Breed1 > PhotoAmt |
| 11. 深入解 释 | Breed1 品种分析 | 数据支持模型判断 |

### 最重要的几条方法原则

1. **先理解数据，再建模** — EDA 不是走流程，是为了回答问题
2. **baseline 先做简单** — 看任务本身有多难，再决定要不要优化
3. **优化不一定成功** — One-Hot 就是例子
4. **问题可以重新定义** — 五分类 → 二分类是最关键的转向
5. **看到反直觉结果，先检查混杂** — Vaccinated 就是教科书级例子
6. **相关性 ≠ 因果** — Breed1 重要不代表品种决定一切
